# EMQA Assignment 1 — starter notebook
**Day-ahead price forecasting and volatility for a Romanian bidding desk** (lectures 1–3).

Read the brief (`A1_brief_EN.pdf`) first. This notebook loads the data and shows the structure you must follow. Replace every `TODO`. Do not change the names of the required objects (`F`, `T5`, `T6`, `T7`, `MY_NUMBERS`): the instructor's checker reads them.

Rules in one line: run top-to-bottom with *Restart & Run All*; no hard-coded dates; no information after gate closure (12:00 CET on day d−1); every number in your memo comes from this notebook.

## 0. Your personal parameters

In [ ]:
STUDENT_ID = 'TODO-your-student-ID'   # team: 'ID1+ID2'
MODE = 'personal'                     # the instructor will also run MODE = 'latest' (unseen hold-out)

import os, sys, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))      # folder that contains emqa_assign.py
import emqa_assign as ea
pd.set_option("display.width", 160)

p = ea.params(STUDENT_ID, 1)
if MODE == "latest":
    p = ea.latest_window(p)
print(json.dumps(p, indent=1))
display(ea.data_version())          # rule R3: record the data version

## 1. Data
`P`: prices (EUR/MWh), `L`: load (MW), `R`: wind + solar (MW); one row per CET delivery day, columns `h00`…`h23` (CET hours). After 2025-10-01 each hourly value is the mean of four quarter-hours.

**Information set (gate closure, 12:00 CET on *d−1*).**
- prices: every day up to *d−1* (the *d−1* auction cleared on *d−2*);
- realised load `L` and wind+solar `R`: only up to day *d−2* (day *d−1* is not yet metered); as a proxy for the TSO's forecasts use `L`, `R` of day *d−2* at the same hour (and *d−3*, *d−7* as further lags);
- transformations and scalers: fitted on these data only.

`ea.leakage_check` enforces this rule: it hides `P` from day *d* and `L`, `R` from day *d−1* onwards.

In [ ]:
P, L, R = ea.panel("RO")
t0, t1 = pd.Timestamp(p["test_start"]), pd.Timestamp(p["test_end"])
test = pd.date_range(t0, t1, freq="D")                                         # evaluation days
fdays = pd.date_range(t0 - pd.Timedelta(days=p["burn_in_days"]), t1, freq="D")  # burn-in + test: forecast ALL of these
print(P.shape, P.index.min().date(), "->", P.index.max().date())
print("test:", test[0].date(), "->", test[-1].date(), "| forecast days incl. burn-in:", len(fdays))
display(P.loc[test[:3]].round(1))

## Task 1 — The test window (table T1 + one figure)

In [ ]:
# TODO: compute on the test window: mean, std, min, max, number of negative hours,
#       number of negative MTUs (rows of ea.quarter_hours("RO"): hourly before 2025-10-01, 15-min after), share of days with 15-min MTU.
T1 = pd.Series(dtype=float)
# TODO: figure: daily base price 2023 -> end of test, test window shaded; y-axis "EUR/MWh".

## Task 2 — Naive benchmarks
Every forecasting function has the same signature `f(P, L, R, day) -> 24 numbers` so that `ea.leakage_check` can test it.

In [ ]:
def naive_weron(P, L, R, day):
    """Mon, Sat, Sun: prices of d-7; Tue-Fri: prices of d-1 (same hour)."""
    # TODO
    raise NotImplementedError

def seasonal_naive(P, L, R, day):
    """prices of d-7, same hour"""
    # TODO
    raise NotImplementedError

## Task 3 — Expert ARX, rolling calibration windows and averaging
Skeleton of the rolling loop (the brief, section *Worked example*, shows one day by hand).

In [ ]:
def arx_forecast(P, L, R, day, T):
    """Expert ARX, one regression per hour h, calibrated on the T days d-T..d-1.
    Regressors (transformed prices, see brief): P[d-1,h], P[d-2,h], P[d-7,h], P[d-1,h23],
    min(P[d-1,:]), max(P[d-1,:]), L[d-2,h], R[d-2,h], Monday, Saturday, Sunday dummies, constant.
    L and R: only rows up to d-2 (day d-1 is not yet metered at gate closure), also for their mean and standard deviation."""
    i = P.index.get_loc(day)
    cal = np.arange(i - T, i)          # calibration TARGET days: rows i-T ... i-1 (all before `day`)
    # 1) variance-stabilising transform with median and MAD of the prices in rows (cal[0]-7) ... (i-1)
    # 2) for each hour h: build X_cal (T x 12) and y_cal (T), least squares
    # 3) build x_day from price rows i-1, i-2, i-7 and L/R row i-2 only; predict; back-transform
    # TODO
    raise NotImplementedError

# F = {"Naive (Weron)": DataFrame(fdays x 24), ..., "ARX(56)": ..., "AW-ARX(6 windows)": ...}
F = {}
# TODO: fill F for every model, for all days in fdays
# ea.leakage_check(lambda P, L, R, d: arx_forecast(P, L, R, d, 56), P, L, R, test[0])

## Task 4 — LEAR (LASSO-estimated autoregression)

In [ ]:
from sklearn.linear_model import LassoLarsIC
def lear_forecast(P, L, R, day, T=p["lear_window"]):
    """247 regressors: prices d-1, d-2, d-3, d-7 (4 x 24), load and wind+solar proxies of d-2, d-3, d-7
    (2 x 3 x 24), 7 day-of-week dummies. One LassoLarsIC(criterion='aic') per hour."""
    # TODO
    raise NotImplementedError

## Task 5 — Accuracy and Diebold–Mariano tests
Required tables: `T5` (one row per model) and two DM p-value matrices.

In [ ]:
T5 = pd.DataFrame(columns=["MAE", "RMSE", "rMAE", "MAE_focus", "RMSE_focus", "rMAE_focus"])   # + per-year columns
# TODO: per calendar year and per quarter; DM (multivariate, |e|, Newey-West variance with 7 lags)

## Task 6 — Probabilistic forecasts (HS and CP)

In [ ]:
QS = np.arange(1, 100) / 100          # the 99 percentiles
# Q[model] : array (n_test_days, 24, 99); errors of the previous p["error_window"] days, same hour
T6 = pd.DataFrame(columns=["APS", "APS_focus", "cov50", "cov90", "cov90_focus", "Kupiec90_focus_p", "width90_focus"])
# TODO

## Task 7 — Volatility (GARCH, GJR-GARCH, EGARCH with Student-t errors)

In [ ]:
from arch import arch_model
daily = ea.daily_prices()["RO"].dropna()
# r_t = 100 * (asinh(P_t) - asinh(P_{t-1}));  remove day-of-week means estimated on the estimation sample only
# estimation sample: [test_start - p["garch_years"] years, test_start);  test: 1-day-ahead variance forecasts
T7 = pd.DataFrame()
# TODO

## Task 8 — Numbers for the memo and hand-reconciliation table
Every number you quote in the memo must be in `MY_NUMBERS` (rule R5).

In [ ]:
MY_NUMBERS = {"params": p}
# TODO: add T1, T5, DM, T6, T7 tables (as dicts), the reconciliation rows for p["check_days"] x p["check_hour"],
#       and your decision.
with open(f"my_numbers_A1_{p['student_id']}_{MODE}.json", "w") as fh:
    json.dump(MY_NUMBERS, fh, indent=1, default=str)

## Self-check before submitting

In [ ]:
checks = {
    "forecasts exist for every model": len(F) >= 10,
    "forecasts cover burn-in + test": all(f.index.equals(fdays) for f in F.values()) if F else False,
    "no missing forecasts": all(not f.isna().any().any() for f in F.values()) if F else False,
    "T5 filled": len(T5) > 0, "T6 filled": len(T6) > 0, "T7 filled": len(T7) > 0,
}
for k, v in checks.items():
    print(("OK   " if v else "FAIL ") + k)